# 04 Feature Engineering

This notebook creates lag and rolling PM2.5 features.

Note:
- these lag features use previous PM2.5 history
- this usually improves R2 a lot compared with weather-only prediction
- the train/test split stays chronological, but now keeps a more proportionate holdout
  so the models do not get starved of training history

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

BASE_DIR = Path.cwd()
AQI_DIR = BASE_DIR / "data" / "AQI"
HTML_DIR = BASE_DIR / "data" / "html_data"
ARTIFACTS_DIR = BASE_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
data = joblib.load(ARTIFACTS_DIR / "pm25_base_model_data.joblib")
data.head()

,Date,Avg_Temperature,Max_Temperature,Min_Temperature,Humidity,Rainfall_Snowmelt,Visibility,Wind_Speed,Max_Sustained_Wind_Speed,City,Month,Year,Day,DayOfWeek,IsWeekend,Season,Daily_PM25
0,2013-01-01,23.4,30.3,19.0,59.0,0.0,6.3,4.3,5.4,Bangalore,1,2013,1,1,0,Winter,282.460976
1,2013-01-02,22.4,30.3,16.9,57.0,0.0,6.9,3.3,7.6,Bangalore,1,2013,2,2,0,Winter,234.543243
2,2013-01-03,24.0,31.8,16.9,51.0,0.0,6.9,2.8,5.4,Bangalore,1,2013,3,3,0,Winter,173.365854
3,2013-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Bangalore,1,2013,4,4,0,Winter,200.244000
4,2013-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Bangalore,1,2013,5,5,1,Winter,213.846725


In [3]:
feature_data = data.copy()

for lag in [1, 2, 3, 5, 7, 10, 14, 21, 30]:
    feature_data[f"PM25_lag_{lag}"] = feature_data["Daily_PM25"].shift(lag)

for window in [3, 5, 7, 10, 14, 21, 30]:
    feature_data[f"PM25_roll_mean_{window}"] = feature_data["Daily_PM25"].shift(1).rolling(window).mean()

feature_data["PM25_change_1"] = feature_data["Daily_PM25"].shift(1) - feature_data["Daily_PM25"].shift(2)
feature_data["PM25_change_3"] = feature_data["Daily_PM25"].shift(1) - feature_data["Daily_PM25"].shift(4)
feature_data["Temp_Range"] = feature_data["Max_Temperature"] - feature_data["Min_Temperature"]
feature_data["Month_sin"] = np.sin(2 * np.pi * feature_data["Month"] / 12)
feature_data["Month_cos"] = np.cos(2 * np.pi * feature_data["Month"] / 12)

required_history_features = [
    "PM25_change_1",
    "PM25_change_3",
]
required_history_features += [f"PM25_lag_{lag}" for lag in [1, 2, 3, 5, 7, 10, 14, 21, 30]]
required_history_features += [f"PM25_roll_mean_{window}" for window in [3, 5, 7, 10, 14, 21, 30]]

feature_data = feature_data.dropna(subset=required_history_features).reset_index(drop=True)

print("Feature engineered dataset shape:", feature_data.shape)
feature_data.head()

Feature engineered dataset shape: (1046, 38)


,Date,Avg_Temperature,Max_Temperature,Min_Temperature,Humidity,Rainfall_Snowmelt,Visibility,Wind_Speed,Max_Sustained_Wind_Speed,City,Month,Year,Day,DayOfWeek,IsWeekend,Season,Daily_PM25,PM25_lag_1,PM25_lag_2,PM25_lag_3,PM25_lag_5,PM25_lag_7,PM25_lag_10,PM25_lag_14,PM25_lag_21,PM25_lag_30,PM25_roll_mean_3,PM25_roll_mean_5,PM25_roll_mean_7,PM25_roll_mean_10,PM25_roll_mean_14,PM25_roll_mean_21,PM25_roll_mean_30,PM25_change_1,PM25_change_3,Temp_Range,Month_sin,Month_cos
0,2013-01-31,22.1,29.8,17.3,54.0,0.0,6.9,5.2,9.4,Bangalore,1,2013,31,3,0,Winter,301.334146,244.021951,242.426829,167.280488,148.017073,217.187805,113.097561,228.875610,150.221951,282.460976,217.909756,189.151220,191.154007,184.633902,165.995645,187.664896,197.454337,1.595122,100.012195,12.5,0.500000,0.866025
1,2013-02-01,21.6,29.2,15.3,51.0,0.0,6.9,3.9,5.4,Bangalore,2,2013,1,4,0,Winter,338.873171,301.334146,244.021951,242.426829,144.009756,175.134146,166.175610,43.417073,118.700000,234.543243,262.594309,219.814634,203.174913,203.457561,171.171254,194.860715,198.083443,57.312195,134.053659,13.9,0.866025,0.500000
2,2013-02-02,21.4,29.2,15.2,45.0,0.0,6.9,2.8,5.4,Bangalore,2,2013,2,5,1,Winter,277.860976,338.873171,301.334146,244.021951,167.280488,148.017073,228.987805,101.029268,211.092683,173.365854,294.743089,258.787317,226.566202,220.727317,192.275261,205.345152,201.561107,37.539024,96.446341,14.0,0.866025,0.500000
3,2013-02-03,21.8,29.6,14.4,37.0,0.0,7.4,2.2,3.5,Bangalore,2,2013,3,6,1,Winter,179.924390,277.860976,338.873171,301.334146,242.426829,144.009756,217.187805,104.278049,267.736585,200.244000,306.022764,280.903415,245.115331,225.614634,204.906098,208.524594,205.044278,-61.012195,33.839024,15.2,0.866025,0.500000
4,2013-02-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Bangalore,2,2013,4,0,0,Winter,171.146341,179.924390,277.860976,338.873171,244.021951,167.280488,175.134146,113.097561,336.803448,213.846725,265.552846,268.402927,250.245993,221.888293,210.309408,204.343061,204.366958,-97.936585,-121.409756,NaN,0.866025,0.500000


In [4]:
feature_columns = [col for col in feature_data.columns if col not in ["Date", "Daily_PM25"]]
target_column = "Daily_PM25"

minimum_train_rows = 650
minimum_test_rows = 250

if len(feature_data) < minimum_train_rows + minimum_test_rows:
    raise ValueError(
        f"Need at least {minimum_train_rows + minimum_test_rows} rows after feature engineering, "
        f"but only found {len(feature_data)}."
    )

test_size = max(minimum_test_rows, int(len(feature_data) * 0.24))
test_size = min(test_size, len(feature_data) - minimum_train_rows)
split_index = len(feature_data) - test_size

train_df = feature_data.iloc[:split_index].copy()
test_df = feature_data.iloc[split_index:].copy()

X_train = train_df[feature_columns]
X_test = test_df[feature_columns]
y_train = train_df[target_column]
y_test = test_df[target_column]

numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [col for col in X_train.columns if col not in numeric_features]

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_features),
    ("cat", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Train rows:", len(train_df))
print("Test rows :", len(test_df))
print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_test shape :", X_test_processed.shape)

Train rows: 795
Test rows : 251
Processed X_train shape: (795, 39)
Processed X_test shape : (251, 39)


In [5]:
joblib.dump(preprocessor, ARTIFACTS_DIR / "pm25_preprocessor.joblib")
joblib.dump(feature_data, ARTIFACTS_DIR / "pm25_feature_data.joblib")
joblib.dump(X_train_processed, ARTIFACTS_DIR / "pm25_X_train_processed.joblib")
joblib.dump(X_test_processed, ARTIFACTS_DIR / "pm25_X_test_processed.joblib")
joblib.dump(y_train.reset_index(drop=True), ARTIFACTS_DIR / "pm25_y_train.joblib")
joblib.dump(y_test.reset_index(drop=True), ARTIFACTS_DIR / "pm25_y_test.joblib")

print("Saved feature engineered artifacts.")

Saved feature engineered artifacts.
